# The Trading Engine

The [lecture](../CHEME-5660-L15a-Lecture-Testing-Trading-Strategies-Fall-2026.ipynb) describes backtests and stress tests in general. Here we look at how our trading engine makes each decision: the estimates, the preference weight, the six triggers, the utility and BAB weighting rules, and the optional market hedge.

___

## Fixed and Updated Estimates

The engine starts from a 2016 to 2024 fit of the single index model. It can keep the estimates fixed or update each stock's baseline and beta at each close. Beta measures a stock's sensitivity to market growth.

The update uses exponentially weighted least squares (EWLS). A day's weight halves every 63 trading days. The starting fit carries the weight of 63 new trading days, and its weight decays at the same rate.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="../figs/Fig-L15a-EWLS-JPM.svg" width="740" style="max-width:100%; height:auto;" alt="The market sensitivity of JPM over 2025 and 2026. A dashed line marks the fixed estimate from the 2016 to 2024 fit, 1.12. The blue updated estimate, with one update per trading day, starts at 1.12, stays near 1.0 through 2025, and drops to about 0.7 in mid-2026. Late in 2026, JPM moves less with the market than the fixed estimate says."/>
    </center>
</div>

In the figure, the fixed beta for JPM is 1.12 and the updated estimate ends at 0.73. The [updating example](../CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb) computes these estimates and follows how they change the basket. The [EWLS derivation](../advanced/ewls/CHEME-5660-L15a-Derivation-EWLS-Update-Fall-2026.ipynb) gives the weighted fit and its daily update.

___

## The Preference Weight

The daily market state $\xi_{t}$ measures the gap between the 21-day and 63-day exponential moving averages of the `SPY` price. With $\bar{S}^{\mathrm{short}}_{t}$ and $\bar{S}^{\mathrm{long}}_{t}$ denoting those averages, the state is given by:

$$
\xi_{t} = -G\left(\frac{\bar{S}^{\mathrm{short}}_{t}}{\bar{S}^{\mathrm{long}}_{t}}-1\right),
$$

where the gain $G>0$ scales $\xi_t$ so that $|\xi_t|\leq 1$ on 99% of training days. The state is positive when the short average lies below the long average.

The single index model gives the expected growth rate $\tilde{g}_{i,t}$ (1/yr) of stock $i$ from its baseline $\alpha_i(t)$ (1/yr), its market sensitivity $\beta_i(t)$, and the 252-day exponential moving average $\tilde{g}_{M,t}$ of market growth (1/yr). The preference weight $\gamma_i(t)$ of this stock is given by:

$$
\begin{aligned}
\tilde{g}_{i,t} &= \alpha_i(t)+\beta_i(t)\,\tilde{g}_{M,t},\\[4pt]
\gamma_i(t) &= \tanh\left(h\,\frac{\tilde{g}_{i,t}}{\max\!\left(|\beta_i(t)|,\,\beta_{\mathrm{floor}}\right)^{\xi_t}}\right),
\end{aligned}
$$

where $h=1$ year makes the argument dimensionless and $\beta_{\mathrm{floor}}=0.1$ keeps the denominator positive. Among stocks with positive expected growth, a positive market state favors lower absolute beta at the same expected growth rate.

The preference weight updates as new prices arrive. The picker uses today's preference weights when a trigger calls for a new basket.

___

## The Six Triggers
A new decision starts only when a trigger fires. After each close, the engine checks six triggers:

| Trigger | Fires when |
|:--|:--|
| Scheduled | 21 trading days have passed since the last decision. |
| Market turn | The market state has changed sign since the last decision. With the hedge on, crossing one also counts. |
| Band exit | Wealth leaves the middle 95% of the forecast that the last decision made for the next 21 trading days. |
| Drawdown | Wealth falls more than 8% below its peak. The peak resets at the drawdown decision. |
| Market sensitivity over its limit | With the hedge on, the portfolio's beta exceeds its limit by more than 0.1. |
| Back from cash | 21 trading days have passed since the last decision and that decision put the portfolio in cash. |

The market-turn trigger uses the market state $\xi_{t}$. A change of sign in $\xi_{t}$ starts a new decision without waiting for the schedule. The schedule then counts from that decision:

<div>
    <center>
        <img class="course-diagram" src="../figs/Fig-L15a-Gamma-Picks.svg" width="860" style="max-width:100%; height:auto;" alt="One time axis with three rows. The top row is a path of the market state xi t, shaded blue where the market is rising and red where it is falling. The middle row is a line with a dot for every trading day: the preference weights gamma update at every close, and alpha and beta also update with updated estimates. The bottom row shows the baskets held, numbered 1 to 7, each picked at one decision and held until the next. Most baskets last 21 trading days. A brace over basket 2 reads: 15 trading days, cut short by the market turn. Where xi t crosses zero, an orange dashed line marks a market turn: the method picks a new basket, and the 21-day count restarts."/>
    </center>
</div>

___

## The Utility Basket and Betting Against Beta

Five weighting rules on the menu use the preference weights in a utility. At each decision, the engine chooses the shares that maximize the utility under the trading limits:

> __Definition (constrained utility allocation):__ On decision day $t$, let $S_{i}$ (USD/share) be the closing price of ticker $i$, $\beta_{i}$ its market sensitivity, $n_{i}^{\text{prev}}$ the shares held before the decision, and $W$ (USD) the wealth to invest. Choose the share counts $n_{i}$ of the $K$ tickers the rule may hold, negative for a short position, and the cash $c\geq 0$ (USD), with weights $w_{i}=n_{i}S_{i}/W$ and a utility $U(n;\gamma)$ from the table below, to solve:
>
> $$
> \boxed{\begin{aligned}
> \operatorname*{maximize}_{n_{1},\ldots,n_{K},\;c\geq 0}\quad & U(n;\gamma)\\
> \text{subject to}\quad & \sum_{i=1}^{K}n_{i}S_{i}+c=W && \text{(budget)}\\
> & |w_{i}|\leq\bar{w}_{i}\quad\text{for each }i && \text{(weight limit)}\\
> & \sqrt{w^{\top}\Sigma_{g}\,w}\leq\sigma_{\text{SPY}} && \text{(volatility)}\\
> & \bar{c}\sum_{i=1}^{K}\left|n_{i}-n_{i}^{\text{prev}}\right|\leq B && \text{(trading cost)}\\
> & \sum_{i=1}^{K}\beta_{i}\,w_{i}\leq\bar{\beta}_{t} && \text{(market sensitivity)}
> \end{aligned}}
> $$
>
> The weight limit $\bar{w}_{i}$ is the largest fraction of the wealth that ticker $i$ may hold, long or short. The volatility limit keeps the estimated volatility of the portfolio at or below that of `SPY` on the training data. Here $\Sigma_{g}$ (1/yr²) is the estimated covariance of the growth rates of the tickers, and $\sigma_{\text{SPY}}$ (1/yr) is the standard deviation of the growth rate of `SPY`.
>
> Each share bought or sold costs $\bar{c}$ (USD/share), an estimate of half the bid-ask spread. The estimated trading cost of one decision may not exceed $B$ (USD). With the hedge on, the market sensitivity of the holdings may not exceed $\bar{\beta}_{t}=\max(0,1-\xi_{t})$.

A numerical solver finds the constrained optimum. Cash can remain when a limit stops further purchases. The five utilities, and the allocations they give before the trading limits, are:

$$
\begin{array}{lcc}
\hline
\text{Utility} & U(n;\gamma) & \text{Allocation before the trading limits} \\[5pt]
\hline
\text{Cobb-Douglas} & \displaystyle\sum_{i}\gamma_{i}\ln n_{i} & w_{i}\propto\gamma_{i} \\[9pt]
\text{CES on shares} & \displaystyle\left(\sum_{i}\gamma_{i}n_{i}^{\rho}\right)^{1/\rho} & n_{i}\propto(\gamma_{i}/S_{i})^{\eta} \\[9pt]
\text{CES on weights} & \displaystyle\left(\sum_{i}\gamma_{i}w_{i}^{\rho}\right)^{1/\rho} & w_{i}\propto\gamma_{i}^{\eta} \\[9pt]
\text{Stone-Geary} & \displaystyle\sum_{i}\gamma_{i}\ln(n_{i}-b_{i}) & (n_{i}-b_{i})S_{i}\propto\gamma_{i} \\[9pt]
\text{130/30 long-short} & \displaystyle\sum_{\gamma_{i}>0}\gamma_{i}\ln n_{i}^{+}+\sum_{\gamma_{i}<0}|\gamma_{i}|\ln n_{i}^{-} & w_{i}\propto\gamma_{i}\text{ on each side} \\[12pt]
\hline
\end{array}
$$

The first four utilities never sell a stock short, and their sums run over the tickers with positive preference weights. The two constant elasticity of substitution (CES) utilities use the elasticity $\eta=0.5$, with $\rho=1-1/\eta$. The Stone-Geary utility keeps a baseline of $b_{i}$ shares of every ticker, worth 1% of the wealth.

The 130/30 rule buys the picked tickers with positive preference weights and sells short up to three tickers per sector with the most negative preference weights. Its long positions $n_{i}^{+}$ may be worth up to 130% of the wealth, and its short positions $n_{i}^{-}$ up to 30%.

The other weighting rules on the menu meet the same trading limits with their own objectives. Like the 130/30 rule, the BAB rule also limits its total long and short positions.

__The utility basket.__ The utility basket weights the picked stocks with one of the first four utilities in the table. The examples start with Cobb-Douglas, as the practicum does.

__Betting against beta.__ The BAB weighting rule ranks the picked stocks by their market sensitivity. The rule buys the lower-beta stocks and sells the higher-beta stocks short, in amounts that make the portfolio's beta zero before the trading limits.

When the market falls, the preference weight tilts the utility basket toward the low-beta stocks that BAB buys:

<div>
    <center>
        <img class="course-diagram" src="../figs/Fig-L15a-BAB-Tilt.svg" width="860" style="max-width:100%; height:auto;" alt="Two panels share the horizontal axis, the market sensitivity beta i, from 0 to 2. The left panel, utility basket, shows the factor that the preference weight applies to the expected growth rate, one over the larger of the absolute value of beta i and beta floor, with that larger value raised to the power xi t. A red curve for a falling market, xi t above 0, is high at low beta and falls below 1 at high beta, labeled favors low beta. A blue line for a rising market, xi t below 0, rises with beta, labeled favors high beta. A gray dashed line at 1 is labeled xi t equals 0, no tilt. The three cross at beta i equals 1. The right panel, betting against beta, shows bars for the weights of eight stocks before the trading limits. A dashed line at the middle rank splits them. The four stocks with low beta have positive weights, labeled long, and the weight is largest for the lowest beta. The four stocks with high beta have negative weights, labeled short."/>
    </center>
</div>

* __Signal:__ Among stocks with the same positive expected growth rate, the preference weight favors stocks that move less with the market when the market falls ($\xi_{t}>0$). When the market rises ($\xi_{t}<0$), the tilt reverses toward the high-beta stocks that BAB sells short.
* __Long-only basket:__ The utility basket never sells a stock short. A stock with a low preference weight gets a small weight or none. With the hedge on, the only short position is `SPY`.
* __Market sensitivity:__ The BAB rule sizes its two sides so that the portfolio's beta is zero before the trading limits. The beta of the utility basket is the weighted average of the betas of the stocks it holds, and cash has a beta of zero.

___

## The Market Hedge

With the optional hedge on, the engine limits the portfolio's market sensitivity at each new decision. Let $w_i(t)$ be the weight of holding $i$ as a fraction of wealth, with a negative weight for a short. The portfolio's market sensitivity $\beta_P(t)$ must satisfy:

$$
\beta_P(t)=\sum_i w_i(t)\,\beta_i(t)\leq \max\!\left(1-\xi_t,\,-0.5\right).
$$

Cash has zero beta. The limit falls as the market state rises.

<div>
    <center>
        <img class="course-diagram" src="../figs/Fig-L15a-Market-Hedge.svg" width="720" style="max-width:100%; height:auto;" alt="A line shows the limit on the market sensitivity of the portfolio, beta P of t, against the market state xi t. The limit falls from 2 at xi t equals minus 1 to 1 at xi t equals 0, the market sensitivity of SPY. It reaches 0 at xi t equals 1, the market sensitivity of betting against beta. Below zero, a red area marks where the method sells SPY short, and the line flattens at a floor of minus 0.5. Arrows from the line point down: beta P of t stays on or below the line. A gray band marks where the absolute value of xi t is at most 1, which holds on 99 percent of training days."/>
    </center>
</div>

- When $\xi_t<0$, the limit is above one. The hedge allows the portfolio to be more sensitive to the market than `SPY`.
- As $\xi_t$ rises from zero to one, the limit drops from one to zero. The engine can put more weight in low-beta stocks or cash. It can also add up to five stocks with zero or negative beta from its hedge pool.
- When $\xi_t>1$, the engine shorts `SPY`. A short worth 20% of the portfolio's value lowers portfolio beta by 0.2. The floor of −0.5 limits the short to half of the portfolio's value.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___